# 2DGS Mitsubishi Pipeline — 품질 저하 원인 진단

**30k iter 학습 결과 PSNR=12.88** 저조 원인을 단계별로 분리 검증하는 노트북.

- Part 0: 셋업 (pipeline_colab.ipynb 와 동일)
- Part 1: 데이터 무결성 (학습 없음)
- Part 2: Oracle 렌더 (학습 없거나 단일 뷰)
- Part 3: 9개 격리 학습 그리드 (각 7k iter)
- Part 4: 결과 집계 + Drive 백업
- Part 5: 로컬 다운로드 지침

모든 산출물은 `/content/diagnosis_results/` → zip → Google Drive → 로컬.

## Part 0. 환경 셋업

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE = '/content/drive/MyDrive/2dgs_diagnosis'
import os
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f'Drive mounted. Save path: {DRIVE_SAVE}')

In [ ]:
import os
ROOT = '/content'
REPO = os.path.join(ROOT, '2d-gaussian-splatting')
DATA = os.path.join(ROOT, 'mitsubishi')
RESULTS = os.path.join(ROOT, 'diagnosis_results')

for sub in ['phase1', 'phase2', 'phase3', 'phase4']:
    os.makedirs(os.path.join(RESULTS, sub), exist_ok=True)
print('Results root:', RESULTS)

In [ ]:
if not os.path.exists(REPO):
    !git clone https://github.com/BAEJUNHAK/2d-gaussian-splatting.git --recursive {REPO}

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}')

In [ ]:
# 의존성 + 서브모듈 빌드 (pipeline_colab.ipynb 동일)
!pip install -q plyfile scikit-image==0.21.0 lpips==0.1.4 trimesh==4.3.2 \
    open3d mediapy==1.1.2 opencv-python tqdm pillow tensorboard

# simple_knn FLT_MAX 패치
knn_cu = os.path.join(REPO, 'submodules', 'simple-knn', 'simple_knn.cu')
with open(knn_cu, 'r') as f:
    src = f.read()
if '#include <cfloat>' not in src:
    src = src.replace('#include "simple_knn.h"', '#include "simple_knn.h"\n#include <cfloat>')
    with open(knn_cu, 'w') as f:
        f.write(src)
    print('Patched simple_knn.cu')

!pip install -q {REPO}/submodules/diff-surfel-rasterization
!pip install -q {REPO}/submodules/simple-knn

In [ ]:
# 데이터 다운로드 (pipeline_colab.ipynb 와 동일)
!pip install -q gdown
import gdown, zipfile, numpy as np
RAW = os.path.join(ROOT, 'raw_data')
os.makedirs(RAW, exist_ok=True)
rgb_zip = os.path.join(RAW, 'dataset.zip')
if not os.path.exists(rgb_zip):
    gdown.download(id='1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB', output=rgb_zip, quiet=False)
depth_zip = os.path.join(RAW, 'depth_data.zip')
if not os.path.exists(depth_zip):
    gdown.download(id='1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ', output=depth_zip, quiet=False)

RGB_DIR = os.path.join(RAW, 'dataset')
DEPTH_DIR = os.path.join(RAW, 'dataset_depth')
if not os.path.exists(RGB_DIR):
    with zipfile.ZipFile(rgb_zip) as z: z.extractall(RAW)
if not os.path.exists(DEPTH_DIR):
    with zipfile.ZipFile(depth_zip) as z: z.extractall(RAW)

all_rgb_files = sorted([f for f in os.listdir(RGB_DIR) if f.startswith('rgb_')])
all_calib_files = sorted([f for f in os.listdir(RGB_DIR) if f.startswith('calib_')])
all_depth_files = sorted([f for f in os.listdir(DEPTH_DIR) if f.startswith('depth_')])

NUM_VIEWS = 50
indices = np.linspace(0, len(all_rgb_files) - 1, NUM_VIEWS, dtype=int)
rgb_files = [all_rgb_files[i] for i in indices]
calib_files = [all_calib_files[i] for i in indices]
depth_files = [all_depth_files[i] for i in indices]
print(f'Total={len(all_rgb_files)}, Sampled={NUM_VIEWS}')

In [ ]:
# 공통 유틸 (calib 파서, COLMAP writer)
import numpy as np, struct
from configparser import ConfigParser
from PIL import Image

def parse_calib_ini(filepath):
    with open(filepath, 'r') as f:
        lines = [l for l in f.readlines() if not l.startswith('#')]
    content = '\n'.join(lines)
    cp = ConfigParser(); cp.read_string(content); sec = 'camera_0'
    k_vals = list(map(float, cp.get(sec, 'k_matrix').split('Matrix:3:3:')[1].split(',')))
    K = np.array(k_vals).reshape(3, 3)
    r_vals = list(map(float, cp.get(sec, 'r_matrix').split('Matrix:3:3:')[1].split(',')))
    R = np.array(r_vals).reshape(3, 3)
    t_vals = list(map(float, cp.get(sec, 't_vector').split('Vector:3:')[1].split(',')))
    t = np.array(t_vals)
    w = int(cp.get(sec, 'width')); h = int(cp.get(sec, 'height'))
    return K, R, t, w, h

def rotmat2qvec(R):
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0: qvec *= -1
    return qvec

def write_cameras_binary(cam_dict, path):
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(cam_dict)))
        for cam_id, cam in cam_dict.items():
            f.write(struct.pack('<iiQQ', cam_id, cam['model_id'], cam['width'], cam['height']))
            for p in cam['params']: f.write(struct.pack('<d', p))

def write_images_binary(img_dict, path):
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(img_dict)))
        for img_id, img in img_dict.items():
            f.write(struct.pack('<i', img_id))
            for q in img['qvec']: f.write(struct.pack('<d', q))
            for t in img['tvec']: f.write(struct.pack('<d', t))
            f.write(struct.pack('<i', img['camera_id']))
            f.write(img['name'].encode('utf-8')); f.write(b'\x00')
            f.write(struct.pack('<Q', 0))

def write_points3D_binary(pts_dict, path):
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(pts_dict)))
        for pt_id, pt in pts_dict.items():
            f.write(struct.pack('<Q', pt_id))
            for x in pt['xyz']: f.write(struct.pack('<d', x))
            for c in pt['rgb']: f.write(struct.pack('<B', int(c)))
            f.write(struct.pack('<d', pt['error']))
            f.write(struct.pack('<Q', 0))

print('Utils loaded.')

## Part 1. 데이터 무결성 진단 (학습 없음)

### 1.1 Intrinsic 단위 확인
**PASS 기준**: cx ≈ w/2, cy ≈ h/2, fx/fy ∈ [100, 10000] (pixel 단위).

In [ ]:
import json, pandas as pd

rows = []
for cal_f in calib_files:
    K, R, t, w, h = parse_calib_ini(os.path.join(RGB_DIR, cal_f))
    rows.append({
        'file': cal_f, 'w': w, 'h': h,
        'fx': K[0,0], 'fy': K[1,1], 'cx': K[0,2], 'cy': K[1,2],
        'cx/w': K[0,2]/w, 'cy/h': K[1,2]/h,
        't_norm': float(np.linalg.norm(t)),
    })
df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS, 'phase1', 'intrinsics.csv'), index=False)

stats = {
    'fx_mean': float(df.fx.mean()), 'fx_std': float(df.fx.std()),
    'fy_mean': float(df.fy.mean()), 'fy_std': float(df.fy.std()),
    'cx_mean': float(df.cx.mean()), 'cy_mean': float(df.cy.mean()),
    'cx_over_w_mean': float(df['cx/w'].mean()),
    'cy_over_h_mean': float(df['cy/h'].mean()),
    't_norm_min': float(df.t_norm.min()),
    't_norm_max': float(df.t_norm.max()),
    't_norm_mean': float(df.t_norm.mean()),
    'w': int(df.w.iloc[0]), 'h': int(df.h.iloc[0]),
    'pass_cx_center': bool(0.45 < df['cx/w'].mean() < 0.55),
    'pass_cy_center': bool(0.45 < df['cy/h'].mean() < 0.55),
    'pass_fx_pixel_scale': bool(100 < df.fx.mean() < 10000),
}
with open(os.path.join(RESULTS, 'phase1', 'intrinsic_report.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))
print('\nSample rows:')
print(df.head(3).to_string())

### 1.2 Extrinsic 4가지 컨벤션 비교 ⭐ (핵심)
- H1: calib R/t = world→cam (COLMAP 기본 — 현재 Colab 가정)
- H2: calib R/t = cam→world
- H3: H1 + Blender OpenGL→COLMAP 축 변환 (Y/Z 부호 반전)
- H4: H2 + 축 변환

각 가정에서 카메라 월드 위치 + view direction 계산 → "look-at 수렴점" 과 "원점 구면성" 을 비교.

In [ ]:
import matplotlib.pyplot as plt

# 모든 calib 파싱
calibs = []
for cal_f in calib_files:
    K, R, t, w, h = parse_calib_ini(os.path.join(RGB_DIR, cal_f))
    calibs.append((R, t))

# Blender OpenGL → COLMAP 축 변환: x 유지, y/z 반전
FLIP = np.diag([1.0, -1.0, -1.0])

def interpret(R, t, hypo):
    """return cam_center (world), view_dir (world, unit)"""
    if hypo == 'H1':        # R,t = w2c
        R_w2c, t_w2c = R, t
    elif hypo == 'H2':      # R,t = c2w → invert
        R_w2c = R.T; t_w2c = -R.T @ t
    elif hypo == 'H3':      # H1 + axis flip on c2w
        R_c2w = FLIP @ R.T; t_c2w = FLIP @ (-R.T @ t)
        R_w2c = R_c2w.T; t_w2c = -R_c2w.T @ t_c2w
    elif hypo == 'H4':      # H2 + axis flip
        R_c2w = FLIP @ R; t_c2w = FLIP @ t
        R_w2c = R_c2w.T; t_w2c = -R_c2w.T @ t_c2w
    C = -R_w2c.T @ t_w2c                      # camera center in world
    view_dir = R_w2c.T @ np.array([0, 0, 1])  # +Z in camera → world
    view_dir = view_dir / (np.linalg.norm(view_dir) + 1e-12)
    return C, view_dir

def lookat_convergence(centers, dirs):
    """Least-squares closest point to all rays C + s·d.
    Returns (p, mean_dist_to_rays)."""
    A = np.zeros((3, 3)); b = np.zeros(3)
    for C, d in zip(centers, dirs):
        P = np.eye(3) - np.outer(d, d)
        A += P; b += P @ C
    p = np.linalg.solve(A + 1e-9*np.eye(3), b)
    dists = []
    for C, d in zip(centers, dirs):
        diff = (np.eye(3) - np.outer(d, d)) @ (p - C)
        dists.append(np.linalg.norm(diff))
    return p, float(np.mean(dists)), float(np.std(dists))

results = {}
fig = plt.figure(figsize=(20, 16))
for i, hypo in enumerate(['H1', 'H2', 'H3', 'H4']):
    centers, dirs = [], []
    for R, t in calibs:
        C, d = interpret(R, t, hypo)
        centers.append(C); dirs.append(d)
    centers = np.array(centers); dirs = np.array(dirs)
    p, conv_mean, conv_std = lookat_convergence(centers, dirs)
    cam_radii = np.linalg.norm(centers - p, axis=1)
    results[hypo] = {
        'lookat_point': p.tolist(),
        'lookat_convergence_mean_dist': conv_mean,
        'lookat_convergence_std_dist': conv_std,
        'cam_radius_mean': float(cam_radii.mean()),
        'cam_radius_std': float(cam_radii.std()),
        'cam_radius_cov': float(cam_radii.std() / (cam_radii.mean() + 1e-9)),
    }
    ax = fig.add_subplot(2, 2, i+1, projection='3d')
    ax.scatter(centers[:,0], centers[:,1], centers[:,2], c='red', s=20, label='cam centers')
    # view direction quiver (short)
    L = cam_radii.mean() * 0.3
    for C, d in zip(centers, dirs):
        ax.plot([C[0], C[0]+d[0]*L], [C[1], C[1]+d[1]*L], [C[2], C[2]+d[2]*L], 'b-', alpha=0.3, linewidth=0.5)
    ax.scatter([p[0]], [p[1]], [p[2]], c='green', s=100, marker='*', label=f'look-at p')
    ax.scatter([0], [0], [0], c='black', s=80, marker='x', label='origin')
    ax.set_title(f'{hypo}\nconv={conv_mean:.3f}±{conv_std:.3f}, '
                 f'R={results[hypo]["cam_radius_mean"]:.2f} (cov={results[hypo]["cam_radius_cov"]:.3f})')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase1', 'extrinsic_hypotheses.png'), dpi=120, bbox_inches='tight')
plt.show()

# 승자: convergence 오차 최소 + cov 최소
score = lambda h: results[h]['lookat_convergence_mean_dist'] + results[h]['cam_radius_cov']
winner = min(results.keys(), key=score)
summary = {'all': results, 'winner': winner, 'score_table': {h: score(h) for h in results}}
with open(os.path.join(RESULTS, 'phase1', 'extrinsic_winner.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\n=== WINNER:', winner, '===')
print(json.dumps(summary, indent=2))

### 1.3 Depth 단위 검증 + re-projection
Phase 1.2 에서 확정된 컨벤션으로 뷰 A → 뷰 B 투영 일관성 확인.

In [ ]:
# 각 hypothesis 별로 reprojection 오차 측정
def build_w2c(R, t, hypo):
    if hypo == 'H1':
        return R, t
    elif hypo == 'H2':
        return R.T, -R.T @ t
    elif hypo == 'H3':
        R_c2w = FLIP @ R.T; t_c2w = FLIP @ (-R.T @ t)
        return R_c2w.T, -R_c2w.T @ t_c2w
    elif hypo == 'H4':
        R_c2w = FLIP @ R; t_c2w = FLIP @ t
        return R_c2w.T, -R_c2w.T @ t_c2w

def reproject_error(hypo, idx_a=0, idx_b=25, n_samples=500):
    # Load views A, B
    K_a, R_a_raw, t_a_raw, w, h = parse_calib_ini(os.path.join(RGB_DIR, calib_files[idx_a]))
    K_b, R_b_raw, t_b_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, calib_files[idx_b]))
    R_a, t_a = build_w2c(R_a_raw, t_a_raw, hypo)
    R_b, t_b = build_w2c(R_b_raw, t_b_raw, hypo)
    depth_a = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[idx_a]))).astype(np.float64) * 0.01
    depth_b = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[idx_b]))).astype(np.float64) * 0.01
    fg_a = depth_a > 0
    vs, us = np.where(fg_a)
    if len(vs) < n_samples:
        return float('nan'), 0
    chosen = np.random.RandomState(0).choice(len(vs), n_samples, replace=False)
    u = us[chosen].astype(np.float64); v = vs[chosen].astype(np.float64)
    z = depth_a[vs[chosen], us[chosen]]
    fx, fy, cx, cy = K_a[0,0], K_a[1,1], K_a[0,2], K_a[1,2]
    # cam A coords
    x_cam = (u - cx) * z / fx; y_cam = (v - cy) * z / fy
    pts_cam_a = np.stack([x_cam, y_cam, z], axis=1)
    # world
    pts_world = (R_a.T @ (pts_cam_a - t_a).T).T
    # cam B
    pts_cam_b = (R_b @ pts_world.T).T + t_b
    fx_b, fy_b, cx_b, cy_b = K_b[0,0], K_b[1,1], K_b[0,2], K_b[1,2]
    # project
    z_b = pts_cam_b[:, 2]
    valid = z_b > 0.01
    u_b = fx_b * pts_cam_b[:, 0] / z_b + cx_b
    v_b = fy_b * pts_cam_b[:, 1] / z_b + cy_b
    in_img = valid & (u_b >= 0) & (u_b < w) & (v_b >= 0) & (v_b < h)
    if in_img.sum() == 0:
        return float('nan'), 0
    # compare depth
    u_i = np.clip(u_b[in_img].astype(int), 0, w-1)
    v_i = np.clip(v_b[in_img].astype(int), 0, h-1)
    gt_z = depth_b[v_i, u_i]
    pred_z = z_b[in_img]
    valid_gt = gt_z > 0
    if valid_gt.sum() == 0:
        return float('nan'), 0
    rel_err = np.abs(pred_z[valid_gt] - gt_z[valid_gt]) / (gt_z[valid_gt] + 1e-9)
    return float(np.median(rel_err)), int(valid_gt.sum())

reproj = {}
for hypo in ['H1', 'H2', 'H3', 'H4']:
    err_list = []
    for pair in [(0, 10), (0, 25), (10, 40), (5, 45)]:
        e, n = reproject_error(hypo, pair[0], pair[1])
        err_list.append({'pair': pair, 'median_rel_err': e, 'n_valid': n})
    reproj[hypo] = err_list
    med = np.nanmedian([x['median_rel_err'] for x in err_list])
    print(f'{hypo}: median rel depth err across pairs = {med:.4f}')

with open(os.path.join(RESULTS, 'phase1', 'reproj_stats.json'), 'w') as f:
    json.dump(reproj, f, indent=2, default=str)
print('\nSaved reproj_stats.json')

### 1.4 PNG 감마 / 색공간

In [ ]:
import struct as _struct

def read_png_chunks(path):
    out = {'chunks': [], 'gAMA': None, 'sRGB': None, 'iCCP': False}
    with open(path, 'rb') as f:
        if f.read(8) != b'\x89PNG\r\n\x1a\n':
            return out
        while True:
            ln_b = f.read(4)
            if len(ln_b) < 4: break
            ln = _struct.unpack('>I', ln_b)[0]
            t = f.read(4).decode('ascii', errors='ignore')
            data = f.read(ln); f.read(4)  # crc
            out['chunks'].append(t)
            if t == 'gAMA':
                out['gAMA'] = _struct.unpack('>I', data)[0] / 100000.0
            elif t == 'sRGB':
                out['sRGB'] = data[0]
            elif t == 'iCCP':
                out['iCCP'] = True
            if t == 'IEND': break
    return out

sample = read_png_chunks(os.path.join(RGB_DIR, rgb_files[0]))
print('PNG chunks in rgb_0:', sample['chunks'])
print('gAMA:', sample['gAMA'], '(None=없음, 0.45=linear, 1.0=sRGB 가정)')
print('sRGB rendering intent:', sample['sRGB'], '(None=없음, 0-3=존재)')
print('iCCP (embedded profile):', sample['iCCP'])

# foreground 밝기 분포
rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_files[0])))
depth_arr = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[0])))
fg_mask = depth_arr > 0
fg_pixels = rgb_arr[fg_mask, :3].astype(np.float64) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ch in enumerate(['R', 'G', 'B']):
    axes[i].hist(fg_pixels[:, i], bins=50, alpha=0.7)
    axes[i].set_title(f'{ch} channel (foreground)')
    axes[i].set_xlabel('value (sRGB 0..1)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase1', 'gamma_histogram.png'), dpi=100)
plt.show()

gamma_report = {
    'png_chunks': sample['chunks'],
    'gAMA': sample['gAMA'],
    'sRGB': sample['sRGB'],
    'iCCP': sample['iCCP'],
    'fg_mean_rgb': fg_pixels.mean(axis=0).tolist(),
    'fg_median_rgb': np.median(fg_pixels, axis=0).tolist(),
}
with open(os.path.join(RESULTS, 'phase1', 'gamma_report.json'), 'w') as f:
    json.dump(gamma_report, f, indent=2)
print(json.dumps(gamma_report, indent=2))

## Part 2. Oracle 렌더 (학습 0 또는 1뷰 오버핏)

### 공통 준비: 현재 Colab 파이프라인의 COLMAP 데이터셋 (baseline = H1 기반) 구축

In [ ]:
import shutil
SCALE_FACTOR = 100.0
CENTER = np.array([0.0, 0.0, 0.0])

def build_colmap_dataset(base_dir, hypo='H1', scale_factor=100.0, rgba=True):
    """Build a COLMAP dataset under base_dir using the given extrinsic hypothesis.
    Re-uses rgb_files, calib_files, depth_files (module globals).
    Returns: (K_ref, fx, fy, cx, cy, w0, h0)
    """
    os.makedirs(os.path.join(base_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, 'sparse', '0'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, 'depth_gt'), exist_ok=True)
    K0, _, _, w0, h0 = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
    fx, fy, cx, cy = K0[0,0], K0[1,1], K0[0,2], K0[1,2]
    cams = {1: {'model_id': 1, 'width': w0, 'height': h0, 'params': [fx, fy, cx, cy]}}
    write_cameras_binary(cams, os.path.join(base_dir, 'sparse', '0', 'cameras.bin'))
    images = {}
    for i, (rgb_f, cal_f, dep_f) in enumerate(zip(rgb_files, calib_files, depth_files)):
        dst = os.path.join(base_dir, 'images', rgb_f)
        if not os.path.exists(dst):
            rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_f)))
            depth_arr = np.array(Image.open(os.path.join(DEPTH_DIR, dep_f)))
            bg_mask = depth_arr == 0
            rgb_arr[bg_mask] = 0
            if rgba:
                alpha = ((~bg_mask) * 255).astype(np.uint8)
                rgba_arr = np.concatenate([rgb_arr, alpha[..., None]], axis=-1)
                Image.fromarray(rgba_arr, mode='RGBA').save(dst)
            else:
                Image.fromarray(rgb_arr[..., :3]).save(dst)
        _, R_raw, t_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, cal_f))
        R_w2c, t_w2c = build_w2c(R_raw, t_raw, hypo)
        qvec = rotmat2qvec(R_w2c)
        tvec = t_w2c / scale_factor
        images[i+1] = {'qvec': qvec, 'tvec': tvec, 'camera_id': 1, 'name': rgb_f}
    write_images_binary(images, os.path.join(base_dir, 'sparse', '0', 'images.bin'))
    for df_ in depth_files:
        src = os.path.join(DEPTH_DIR, df_); dst = os.path.join(base_dir, 'depth_gt', df_)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    return fx, fy, cx, cy, w0, h0

def build_pcd_from_depth(base_dir, hypo, scale_factor=100.0, per_view=2000, stride=10):
    K0, _, _, _, _ = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
    fx, fy, cx, cy = K0[0,0], K0[1,1], K0[0,2], K0[1,2]
    all_pts = []; all_cols = []
    for idx in range(0, len(rgb_files), stride):
        depth_raw = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[idx]))).astype(np.float64)
        depth = depth_raw * 0.01 / scale_factor
        rgb = np.array(Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3]
        fg_v, fg_u = np.where(depth_raw > 0)
        if len(fg_v) == 0: continue
        n = min(per_view, len(fg_v))
        chosen = np.random.RandomState(idx).choice(len(fg_v), n, replace=False)
        u = fg_u[chosen].astype(np.float64); v = fg_v[chosen].astype(np.float64)
        z = depth[fg_v[chosen], fg_u[chosen]]
        x = (u - cx) * z / fx; y = (v - cy) * z / fy
        pts_cam = np.stack([x, y, z], axis=1)
        _, R_raw, t_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, calib_files[idx]))
        R_w2c, t_w2c = build_w2c(R_raw, t_raw, hypo)
        t_norm = t_w2c / scale_factor
        pts_world = (R_w2c.T @ (pts_cam - t_norm).T).T
        cols = rgb[fg_v[chosen], fg_u[chosen], :3]
        all_pts.append(pts_world); all_cols.append(cols)
    all_pts = np.concatenate(all_pts, axis=0); all_cols = np.concatenate(all_cols, axis=0)
    # voxel downsampling
    vox = 0.01 if scale_factor >= 10 else 1.0
    coords = np.floor(all_pts / vox).astype(np.int64)
    _, uniq = np.unique(coords, axis=0, return_index=True)
    all_pts = all_pts[uniq]; all_cols = all_cols[uniq]
    pts_dict = {i+1: {'xyz': p, 'rgb': c, 'error': 0.0} for i, (p, c) in enumerate(zip(all_pts, all_cols))}
    write_points3D_binary(pts_dict, os.path.join(base_dir, 'sparse', '0', 'points3D.bin'))
    return len(all_pts)

# Baseline H1 데이터셋 (기존 Colab 과 동일)
DATA_H1 = os.path.join(ROOT, 'mitsubishi_H1')
if not os.path.exists(os.path.join(DATA_H1, 'sparse', '0', 'images.bin')):
    build_colmap_dataset(DATA_H1, 'H1', 100.0, rgba=True)
    n_pts = build_pcd_from_depth(DATA_H1, 'H1', 100.0)
    print(f'DATA_H1 built: {n_pts} points')
else:
    print('DATA_H1 already exists.')

### 2.1 Oracle 렌더 — GT depth 를 surfel 로 놓고 4 hypothesis 별 단일 forward

In [ ]:
import sys
sys.path.insert(0, REPO)
import torch, math
from PIL import Image as _Image

# 모듈 임포트
from scene.colmap_loader import qvec2rotmat as _q2r
from diff_surfel_rasterization import GaussianRasterizationSettings, GaussianRasterizer
from utils.graphics_utils import getWorld2View2, getProjectionMatrix, focal2fov

@torch.no_grad()
def render_oracle(hypo, idx_list=[0, 5, 10, 20, 30], scale_factor=100.0,
                  surfel_scale=0.005, samples_per_view=20000):
    """GT depth → surfel → CUDA raster → compare to GT RGB."""
    K0, _, _, w0, h0 = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
    fx_, fy_, cx_, cy_ = K0[0,0], K0[1,1], K0[0,2], K0[1,2]
    # collect points from a few views (stride=5 to save memory)
    all_pts = []; all_cols = []
    for idx in range(0, len(rgb_files), 5):
        depth_raw = np.array(_Image.open(os.path.join(DEPTH_DIR, depth_files[idx]))).astype(np.float64)
        depth = depth_raw * 0.01 / scale_factor
        rgb = np.array(_Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3]
        fg_v, fg_u = np.where(depth_raw > 0)
        if len(fg_v) == 0: continue
        n = min(samples_per_view, len(fg_v))
        chosen = np.random.RandomState(idx).choice(len(fg_v), n, replace=False)
        u = fg_u[chosen].astype(np.float64); v = fg_v[chosen].astype(np.float64)
        z = depth[fg_v[chosen], fg_u[chosen]]
        x = (u - cx_) * z / fx_; y = (v - cy_) * z / fy_
        pts_cam = np.stack([x, y, z], axis=1)
        _, R_raw, t_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, calib_files[idx]))
        R_w2c, t_w2c = build_w2c(R_raw, t_raw, hypo)
        t_norm = t_w2c / scale_factor
        pts_world = (R_w2c.T @ (pts_cam - t_norm).T).T
        all_pts.append(pts_world); all_cols.append(rgb[fg_v[chosen], fg_u[chosen], :3])
    pts = np.concatenate(all_pts).astype(np.float32)
    cols = np.concatenate(all_cols).astype(np.float32) / 255.0
    N = pts.shape[0]

    means3D = torch.from_numpy(pts).cuda()
    colors_precomp = torch.from_numpy(cols).cuda()
    opacities = torch.ones((N, 1), device='cuda') * 0.95
    scales = torch.ones((N, 2), device='cuda') * surfel_scale
    rot = torch.zeros((N, 4), device='cuda'); rot[:, 0] = 1.0  # identity quat
    means2D = torch.zeros_like(means3D, requires_grad=False)

    results = []
    bg = torch.tensor([0.0, 0.0, 0.0], device='cuda')
    for idx in idx_list:
        _, R_raw, t_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, calib_files[idx]))
        R_w2c, t_w2c = build_w2c(R_raw, t_raw, hypo)
        t_norm = t_w2c / scale_factor
        # 2DGS Camera convention: readColmap does R = qvec2rotmat(qvec).T (store c2w rotation)
        # and getWorld2View2 re-applies R.T so Rt[:3,:3] = R_w2c. Mimic:
        R_for_gw2v = R_w2c.T
        wv = getWorld2View2(R_for_gw2v, t_norm)
        wv_t = torch.from_numpy(wv).float().transpose(0, 1).cuda()
        FoVx = focal2fov(fx_, w0); FoVy = focal2fov(fy_, h0)
        proj = getProjectionMatrix(0.01, 100.0, FoVx, FoVy).transpose(0,1).cuda()
        full_proj = (wv_t.unsqueeze(0).bmm(proj.unsqueeze(0))).squeeze(0)
        cam_center = wv_t.inverse()[3, :3]
        rs = GaussianRasterizationSettings(
            image_height=h0, image_width=w0,
            tanfovx=math.tan(FoVx*0.5), tanfovy=math.tan(FoVy*0.5),
            bg=bg, scale_modifier=1.0,
            viewmatrix=wv_t, projmatrix=full_proj,
            sh_degree=0, campos=cam_center, prefiltered=False, debug=False,
        )
        rasterizer = GaussianRasterizer(raster_settings=rs)
        rendered, radii, _ = rasterizer(
            means3D=means3D, means2D=means2D,
            shs=None, colors_precomp=colors_precomp,
            opacities=opacities, scales=scales, rotations=rot, cov3D_precomp=None,
        )
        img = rendered.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        gt = np.array(_Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3] / 255.0
        L1 = float(np.abs(img - gt).mean())
        mse = float(((img - gt)**2).mean())
        psnr = float(20 * np.log10(1.0 / (np.sqrt(mse) + 1e-9)))
        results.append({'idx': idx, 'L1': L1, 'PSNR': psnr, 'rendered': img, 'gt': gt})
    return results, N

oracle_summary = {}
fig, axes = plt.subplots(4, 5, figsize=(22, 18))
idx_list = [0, 5, 10, 20, 30]
for row, hypo in enumerate(['H1', 'H2', 'H3', 'H4']):
    try:
        res, n_pts = render_oracle(hypo, idx_list=idx_list)
        oracle_summary[hypo] = {
            'n_pts': n_pts,
            'mean_L1': float(np.mean([r['L1'] for r in res])),
            'mean_PSNR': float(np.mean([r['PSNR'] for r in res])),
            'per_view': [{'idx': r['idx'], 'L1': r['L1'], 'PSNR': r['PSNR']} for r in res],
        }
        for col, r in enumerate(res):
            axes[row, col].imshow(r['rendered'])
            axes[row, col].set_title(f"{hypo} v{r['idx']} PSNR={r['PSNR']:.1f}", fontsize=9)
            axes[row, col].axis('off')
        print(f"{hypo}: mean PSNR={oracle_summary[hypo]['mean_PSNR']:.2f}, L1={oracle_summary[hypo]['mean_L1']:.4f}")
    except Exception as e:
        print(f'{hypo} failed: {e}')
        oracle_summary[hypo] = {'error': str(e)}

plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase2', 'oracle_renders.png'), dpi=100, bbox_inches='tight')
plt.show()

# GT row
fig_gt, axes_gt = plt.subplots(1, 5, figsize=(22, 4))
for col, idx in enumerate(idx_list):
    axes_gt[col].imshow(np.array(_Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3])
    axes_gt[col].set_title(f'GT v{idx}'); axes_gt[col].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase2', 'oracle_gt.png'), dpi=100, bbox_inches='tight')
plt.show()

with open(os.path.join(RESULTS, 'phase2', 'oracle_metrics.json'), 'w') as f:
    json.dump(oracle_summary, f, indent=2)
print('\nOracle best hypothesis:',
      max({k:v for k,v in oracle_summary.items() if 'mean_PSNR' in v}.items(),
          key=lambda kv: kv[1]['mean_PSNR'], default=('none', {})))

### 2.2 단일 뷰 오버핏 (학습 루프 정상성)
1개 뷰만으로 train.py 3000 iter — PSNR>35 예상.

In [ ]:
# single-view 데이터셋 구축: H1 의 1번 뷰만
DATA_SV = os.path.join(ROOT, 'mitsubishi_singleview')
if not os.path.exists(DATA_SV):
    os.makedirs(os.path.join(DATA_SV, 'images'))
    os.makedirs(os.path.join(DATA_SV, 'sparse', '0'))
    # use only first view
    src = os.path.join(DATA_H1, 'images', rgb_files[0])
    shutil.copy2(src, os.path.join(DATA_SV, 'images', rgb_files[0]))
    # Write cameras.bin / images.bin / points3D.bin for single view
    K0, R_raw, t_raw, w0, h0 = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
    fx, fy, cx, cy = K0[0,0], K0[1,1], K0[0,2], K0[1,2]
    cams = {1: {'model_id': 1, 'width': w0, 'height': h0, 'params': [fx, fy, cx, cy]}}
    write_cameras_binary(cams, os.path.join(DATA_SV, 'sparse', '0', 'cameras.bin'))
    R_w2c, t_w2c = build_w2c(R_raw, t_raw, 'H1')
    imgs = {1: {'qvec': rotmat2qvec(R_w2c), 'tvec': t_w2c / 100.0, 'camera_id': 1, 'name': rgb_files[0]}}
    write_images_binary(imgs, os.path.join(DATA_SV, 'sparse', '0', 'images.bin'))
    # Use a tiny random init PCD
    pts_dict = {i+1: {'xyz': np.random.RandomState(0).randn(3) * 0.5, 'rgb': [128, 128, 128], 'error': 0.0} for i in range(5000)}
    write_points3D_binary(pts_dict, os.path.join(DATA_SV, 'sparse', '0', 'points3D.bin'))
print('Single-view dataset ready.')

OUT_SV = os.path.join(ROOT, 'output_singleview')
!rm -rf {OUT_SV}
!cd {REPO} && python train.py \
    -s {DATA_SV} -m {OUT_SV} \
    --sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.0 \
    --depth_ratio 1.0 --iterations 3000 \
    --test_iterations 3000 --save_iterations 3000 \
    --quiet 2>&1 | tail -20

# 최종 render
!cd {REPO} && python render.py -m {OUT_SV} -s {DATA_SV} \
    --depth_ratio 1.0 --skip_test --skip_mesh --quiet 2>&1 | tail -5

# copy to results
import glob
sv_renders = glob.glob(os.path.join(OUT_SV, 'train', 'ours_*', 'renders', '*.png'))
sv_out_dir = os.path.join(RESULTS, 'phase2', 'single_view')
os.makedirs(sv_out_dir, exist_ok=True)
for f in sv_renders:
    shutil.copy2(f, sv_out_dir)
# measure PSNR
if sv_renders:
    rendered = np.array(_Image.open(sv_renders[0]))[..., :3] / 255.0
    gt = np.array(_Image.open(os.path.join(DATA_SV, 'images', rgb_files[0])))[..., :3] / 255.0
    mse = ((rendered - gt)**2).mean()
    psnr_sv = float(20 * np.log10(1.0 / (np.sqrt(mse)+1e-9)))
else:
    psnr_sv = float('nan')
with open(os.path.join(RESULTS, 'phase2', 'single_view_metrics.json'), 'w') as f:
    json.dump({'iters': 3000, 'final_PSNR': psnr_sv}, f, indent=2)
print(f'Single-view overfit PSNR @ 3000 = {psnr_sv:.2f} (expect >30 if loop healthy)')

## Part 3. 격리 학습 그리드 (9 실험 × 7000 iter)
각 실험당 ~5분, 총 ~45분.

In [ ]:
# 추가 데이터셋 구축: no-scale (H1, scale=1), blender-json 경로, random-init
DATA_NOSCALE = os.path.join(ROOT, 'mitsubishi_noscale')
if not os.path.exists(os.path.join(DATA_NOSCALE, 'sparse', '0', 'images.bin')):
    build_colmap_dataset(DATA_NOSCALE, 'H1', scale_factor=1.0, rgba=True)
    n_pts = build_pcd_from_depth(DATA_NOSCALE, 'H1', scale_factor=1.0)
    print(f'DATA_NOSCALE: {n_pts} pts')

DATA_RANDOM = os.path.join(ROOT, 'mitsubishi_random_init')
if not os.path.exists(os.path.join(DATA_RANDOM, 'sparse', '0', 'images.bin')):
    build_colmap_dataset(DATA_RANDOM, 'H1', scale_factor=100.0, rgba=True)
    # Random 100k points in [-2, 2]^3 (scene extent)
    pts = np.random.RandomState(0).uniform(-2, 2, size=(100000, 3))
    cols = np.random.RandomState(1).uniform(0, 255, size=(100000, 3))
    pts_dict = {i+1: {'xyz': pts[i], 'rgb': cols[i], 'error': 0.0} for i in range(100000)}
    write_points3D_binary(pts_dict, os.path.join(DATA_RANDOM, 'sparse', '0', 'points3D.bin'))
    print('DATA_RANDOM: 100k random pts')

# Blender-style JSON 경로
DATA_BLENDER = os.path.join(ROOT, 'mitsubishi_blender')
if not os.path.exists(os.path.join(DATA_BLENDER, 'transforms_train.json')):
    os.makedirs(DATA_BLENDER, exist_ok=True)
    os.makedirs(os.path.join(DATA_BLENDER, 'train'), exist_ok=True)
    os.makedirs(os.path.join(DATA_BLENDER, 'test'), exist_ok=True)
    K0, _, _, w0, h0 = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
    fx = K0[0,0]
    fov_x = float(2 * math.atan(w0 / (2 * fx)))
    # eval=True 의 llffhold=8 로 분리
    train_frames = []; test_frames = []
    for i, (rgb_f, cal_f) in enumerate(zip(rgb_files, calib_files)):
        _, R_raw, t_raw, _, _ = parse_calib_ini(os.path.join(RGB_DIR, cal_f))
        # readCamerasFromTransforms 은 c2w 받아서 c2w[:3,1:3]*=-1 적용 후 w2c.
        # 여기서는 H1(=w2c) 가정 → c2w_inv → axis flip 역연산.
        R_w2c = R_raw; t_w2c = t_raw / 100.0
        R_c2w = R_w2c.T; t_c2w = -R_w2c.T @ t_w2c
        c2w = np.eye(4); c2w[:3,:3] = R_c2w; c2w[:3, 3] = t_c2w
        # reverse the flip the loader will apply (c2w[:3,1:3] *= -1)
        c2w[:3, 1:3] *= -1
        split = 'test' if i % 8 == 0 else 'train'
        dest_dir = os.path.join(DATA_BLENDER, split)
        # copy RGB (no alpha — readCamerasFromTransforms composites against bg)
        src_rgba = os.path.join(DATA_H1, 'images', rgb_f)
        shutil.copy2(src_rgba, os.path.join(dest_dir, os.path.basename(rgb_f)))
        frame = {'file_path': f'./{split}/' + os.path.basename(rgb_f).replace('.png',''),
                 'transform_matrix': c2w.tolist()}
        (test_frames if split == 'test' else train_frames).append(frame)
    for split, frames in [('train', train_frames), ('test', test_frames)]:
        with open(os.path.join(DATA_BLENDER, f'transforms_{split}.json'), 'w') as f:
            json.dump({'camera_angle_x': fov_x, 'frames': frames}, f, indent=2)
    print(f'DATA_BLENDER built: train={len(train_frames)}, test={len(test_frames)}')

print('All datasets ready.')

In [ ]:
# train.py 패치 유틸 (on/off)
train_py_path = os.path.join(REPO, 'train.py')
with open(train_py_path, 'r') as f:
    _train_original = f.read()

_patched_loss = '''        gt_image = viewpoint_cam.original_image.cuda()
        # [mask-aware L1] gt_alpha_mask가 있으면 foreground만 L1 계산
        gt_alpha_mask = viewpoint_cam.gt_alpha_mask
        if gt_alpha_mask is not None:
            gt_alpha_mask = gt_alpha_mask.cuda()
            diff = torch.abs(image - gt_image).mean(0, keepdim=True)
            Ll1 = (diff * gt_alpha_mask).sum() / (gt_alpha_mask.sum() + 1e-8)
        else:
            Ll1 = l1_loss(image, gt_image)
        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))'''

_orig_loss = '''        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)
        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))'''

def set_train_patch(enable_mask_L1: bool):
    with open(train_py_path, 'r') as f:
        src = f.read()
    if enable_mask_L1:
        if _orig_loss in src:
            src = src.replace(_orig_loss, _patched_loss)
        elif _patched_loss in src:
            pass
        else:
            print('WARN: no known loss block found')
    else:
        if _patched_loss in src:
            src = src.replace(_patched_loss, _orig_loss)
        elif _orig_loss in src:
            pass
    with open(train_py_path, 'w') as f:
        f.write(src)

print('Patch toggle ready.')

In [ ]:
EXPERIMENTS = [
    {'name': '3.1_baseline',       'data': DATA_H1,       'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0',  'mask_patch': True,  'eval_flag': True},
    {'name': '3.2_no_scale_norm',  'data': DATA_NOSCALE,  'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0',  'mask_patch': True,  'eval_flag': True},
    {'name': '3.3_no_mask_L1',     'data': DATA_H1,       'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0',  'mask_patch': False, 'eval_flag': True},
    {'name': '3.4_white_bg',       'data': DATA_H1,       'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0 --white_background', 'mask_patch': True, 'eval_flag': True},
    {'name': '3.5_no_reg',         'data': DATA_H1,       'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.0 --depth_ratio 1.0',  'mask_patch': True,  'eval_flag': True},
    {'name': '3.6_strong_dist',    'data': DATA_H1,       'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 1000 --depth_ratio 1.0', 'mask_patch': True,  'eval_flag': True},
    {'name': '3.7_sh3',            'data': DATA_H1,       'extra': '--sh_degree 3 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0',  'mask_patch': True,  'eval_flag': True},
    {'name': '3.8_random_init',    'data': DATA_RANDOM,   'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0',  'mask_patch': True,  'eval_flag': True},
    {'name': '3.9_blender_path',   'data': DATA_BLENDER,  'extra': '--sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 --depth_ratio 1.0 --white_background', 'mask_patch': False, 'eval_flag': True},
]

ITER_PER_EXP = 7000
TEST_ITER = 7000
grid_results = []

for exp in EXPERIMENTS:
    name = exp['name']
    print('\n' + '='*60)
    print(f"Running {name}")
    print('='*60)
    out = os.path.join(ROOT, f'output_{name}')
    !rm -rf {out}
    set_train_patch(exp['mask_patch'])
    eval_str = '--eval' if exp['eval_flag'] else ''
    cmd = (f'cd {REPO} && python train.py -s {exp["data"]} -m {out} {eval_str} '
           f"{exp['extra']} --iterations {ITER_PER_EXP} "
           f'--test_iterations {TEST_ITER} --save_iterations {TEST_ITER} --quiet')
    print('CMD:', cmd)
    t0 = __import__('time').time()
    !{cmd} 2>&1 | tail -15
    train_time = __import__('time').time() - t0

    # render test
    !cd {REPO} && python render.py -m {out} -s {exp['data']} \
        --depth_ratio 1.0 --skip_train --skip_mesh --quiet 2>&1 | tail -3
    # metrics
    !cd {REPO} && python metrics.py -m {out} 2>&1 | tail -10

    # parse results
    res_path = os.path.join(out, 'results.json')
    entry = {'name': name, 'train_time_sec': train_time}
    if os.path.exists(res_path):
        with open(res_path) as f:
            r = json.load(f)
        method_key = list(r.keys())[0] if r else None
        if method_key:
            entry.update({'PSNR': r[method_key].get('PSNR'),
                          'SSIM': r[method_key].get('SSIM'),
                          'LPIPS': r[method_key].get('LPIPS')})
    # copy sample renders + metrics to results dir
    dst_dir = os.path.join(RESULTS, 'phase3', name)
    os.makedirs(dst_dir, exist_ok=True)
    for src_name in ['results.json', 'per_view.json', 'cfg_args']:
        src = os.path.join(out, src_name)
        if os.path.exists(src): shutil.copy2(src, dst_dir)
    # sample 3 test renders
    test_render_dirs = sorted(glob.glob(os.path.join(out, 'test', 'ours_*', 'renders')))
    if test_render_dirs:
        renders = sorted(glob.glob(os.path.join(test_render_dirs[-1], '*.png')))[:3]
        for rf in renders:
            shutil.copy2(rf, dst_dir)
    # 1개 gt
    test_gt_dirs = sorted(glob.glob(os.path.join(out, 'test', 'ours_*', 'gt')))
    if test_gt_dirs:
        gts = sorted(glob.glob(os.path.join(test_gt_dirs[-1], '*.png')))[:1]
        for gf in gts:
            shutil.copy2(gf, os.path.join(dst_dir, 'gt_' + os.path.basename(gf)))
    with open(os.path.join(dst_dir, 'entry.json'), 'w') as f:
        json.dump(entry, f, indent=2)
    grid_results.append(entry)
    print(f"{name}: PSNR={entry.get('PSNR')}, time={train_time:.1f}s")

# 복원 (mask patch on by default)
set_train_patch(True)

with open(os.path.join(RESULTS, 'phase3', 'grid_summary.json'), 'w') as f:
    json.dump(grid_results, f, indent=2)
print('\n=== GRID COMPLETE ===')
print(json.dumps(grid_results, indent=2))

## Part 4. 결과 집계 + Drive 백업

In [ ]:
# 4.1 PSNR 비교 바 차트
names = [e['name'] for e in grid_results]
psnrs = [e.get('PSNR') or 0 for e in grid_results]
fig, ax = plt.subplots(figsize=(12, 5))
colors_list = ['steelblue' if p < 20 else 'tomato' if p < 25 else 'seagreen' for p in psnrs]
bars = ax.bar(names, psnrs, color=colors_list)
for b, p in zip(bars, psnrs):
    ax.text(b.get_x()+b.get_width()/2, p+0.3, f'{p:.1f}', ha='center', fontsize=9)
ax.axhline(y=12.88, color='gray', linestyle='--', label='Current (12.88)')
ax.axhline(y=25, color='green', linestyle='--', label='Healthy threshold')
ax.set_ylabel('Test PSNR @ 7k iter')
ax.set_title('Phase 3 Grid — PSNR by experiment')
plt.xticks(rotation=30, ha='right'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase4', 'psnr_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 콜라주: 각 실험의 첫 번째 test render
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
for i, entry in enumerate(grid_results):
    ax = axes[i//3, i%3]
    exp_dir = os.path.join(RESULTS, 'phase3', entry['name'])
    renders = sorted(glob.glob(os.path.join(exp_dir, '0*.png')))
    if renders:
        img = np.array(_Image.open(renders[0]))
        ax.imshow(img)
    ax.set_title(f"{entry['name']}\nPSNR={entry.get('PSNR', 'NA'):.2f}" if isinstance(entry.get('PSNR'), (int, float)) else entry['name'],
                 fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'phase4', 'rendered_collage.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 최종 summary.md
def fmt_num(v, spec='.2f'):
    if isinstance(v, (int, float)):
        return format(v, spec)
    return 'NA'

lines = []
lines.append('# 2DGS Mitsubishi Pipeline Diagnosis — Summary\n\n')

lines.append('## Phase 1 — Data integrity\n')
with open(os.path.join(RESULTS, 'phase1', 'intrinsic_report.json')) as f:
    ir = json.load(f)
lines.append('- Intrinsics: fx={}±{}, cx/w={}, cy/h={}\n'.format(
    fmt_num(ir['fx_mean'], '.1f'), fmt_num(ir['fx_std'], '.1f'),
    fmt_num(ir['cx_over_w_mean'], '.3f'), fmt_num(ir['cy_over_h_mean'], '.3f')))
lines.append('- t_norm range: [{}, {}]\n'.format(
    fmt_num(ir['t_norm_min']), fmt_num(ir['t_norm_max'])))
with open(os.path.join(RESULTS, 'phase1', 'extrinsic_winner.json')) as f:
    ew = json.load(f)
lines.append('- **Extrinsic winner (look-at score)**: {}\n'.format(ew['winner']))
for h, s in ew['score_table'].items():
    lines.append('  - {}: score={}\n'.format(h, fmt_num(s, '.4f')))

lines.append('\n## Phase 2 — Oracle render\n')
with open(os.path.join(RESULTS, 'phase2', 'oracle_metrics.json')) as f:
    om = json.load(f)
for h, v in om.items():
    if 'mean_PSNR' in v:
        lines.append('- {}: PSNR={}, L1={}\n'.format(
            h, fmt_num(v['mean_PSNR']), fmt_num(v['mean_L1'], '.4f')))
with open(os.path.join(RESULTS, 'phase2', 'single_view_metrics.json')) as f:
    sv = json.load(f)
lines.append('- Single-view overfit @ 3k iter: PSNR={}\n'.format(fmt_num(sv['final_PSNR'])))

lines.append('\n## Phase 3 — Grid (7k iter each)\n')
lines.append('| # | Name | PSNR | SSIM | LPIPS | Time (s) |\n')
lines.append('|---|---|---|---|---|---|\n')
for i, e in enumerate(grid_results):
    lines.append('| {} | {} | {} | {} | {} | {} |\n'.format(
        i+1,
        e.get('name', 'NA'),
        fmt_num(e.get('PSNR')),
        fmt_num(e.get('SSIM'), '.4f'),
        fmt_num(e.get('LPIPS'), '.4f'),
        fmt_num(e.get('train_time_sec'), '.1f'),
    ))
best = max((e for e in grid_results if isinstance(e.get('PSNR'), (int, float))),
           key=lambda e: e['PSNR'], default=None)
if best:
    lines.append('\n**Best experiment**: {} (PSNR={})\n'.format(
        best['name'], fmt_num(best['PSNR'])))
lines.append('\n**Baseline (Colab 30k run)**: PSNR=12.88 → target to beat\n')
with open(os.path.join(RESULTS, 'summary.md'), 'w') as f:
    f.writelines(lines)
print(''.join(lines))

In [ ]:
# 4.4 zip + Drive 백업
ZIP_PATH = '/tmp/diagnosis_results.zip'
!rm -f {ZIP_PATH}
!cd /content && zip -qr {ZIP_PATH} diagnosis_results
!ls -lh {ZIP_PATH}
!cp {ZIP_PATH} {DRIVE_SAVE}/
print(f'Uploaded to Drive: {DRIVE_SAVE}/diagnosis_results.zip')

## Part 5. 로컬 다운로드 지침

1. Google Drive 웹에서 `MyDrive/2dgs_diagnosis/diagnosis_results.zip` 다운로드
2. 로컬에 해제:
   ```bash
   cd /Users/baejunhak/labatory/2d-gaussian-splatting
   unzip ~/Downloads/diagnosis_results.zip
   ```
3. 이 파일들을 확인/공유:
   - `diagnosis_results/summary.md`  ← **최우선**
   - `diagnosis_results/phase1/extrinsic_hypotheses.png` + `extrinsic_winner.json`
   - `diagnosis_results/phase2/oracle_renders.png` + `oracle_gt.png` + `oracle_metrics.json`
   - `diagnosis_results/phase2/single_view_metrics.json`
   - `diagnosis_results/phase4/psnr_comparison.png`
   - `diagnosis_results/phase4/rendered_collage.png`
   - `diagnosis_results/phase3/*/entry.json` (실험별 상세)

다음 턴에서 주요 이미지/JSON 을 붙여주시면 원인을 확정하고 수정안을 제시합니다.